# Data Profiling Report
## Tableau Prep Cleaning Exercise

Pre-cleaning data profiling only. No transformations are applied here.

### Contents
1. Load & Overview
2. Missing Value Analysis
3. Data Type Analysis
4. Duplicate Detection
5. Cardinality Analysis
6. Value Distribution Analysis
7. Text Format & Inconsistency Analysis
8. Date Format Analysis
9. Numeric Validity & Outlier Analysis
10. Column Analysis (Identifiers, Patterns, Domains)
11. Structure Analysis (Functional Dependencies, Calculated Fields)
12. Cleaning Rules Summary

In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("../../data/tableau_prep_cleaning_exercise_clean.csv")
df.shape

(91, 20)

In [2]:
# Preview first 10 rows
df.head(10)

,Sales_MinMax,Age_Group,Customer_Segment,Sales,Notes,Payment_Method,Satisfaction_Score,Age,Ship_Date,Order_Date,Unit_Price,Order_ID,Customer_ID,Customer_Name,Email,Country,City,Product_Category,Quantity,Discount
0,0.306252,65+,Consumer,1283.94,missing phone,Bank Transfer,3,69,25/01/2026,20/01/2026,142.66,O1091,C018,Carlos Garcia,carlos.garcia@example.com,France,Lyon,Home,10,0.10
1,0.110171,65+,Corporate,480.83,missing phone,Paypal,3,45,29/01/2026,26/01/2026,188.56,O1028,C040,Laura Rossi,laura.rossi@example.com,Germany,Karlsruhe,Furniture,3,0.15
2,0.087878,55–64,Consumer,389.52,ok,Credit Card,4,64,10/01/2026,06/01/2026,129.84,O1079,C047,John Doe,john.doe@example.com,Germany,Mannheim,Furniture,4,0.25
3,0.518742,35–44,Consumer,2154.26,check address,Paypal,3,36,12/02/2026,03/02/2026,188.97,O1063,C024,Mehdi Karimi,mehdi.karimi@example.com,Spain,Madrid,Home,12,0.05
4,0.017660,55–64,Consumer,101.92,none,Credit Card,2,62,01/05/2026,26/04/2026,25.48,O1082,C014,Nina Weber,nina.weber@example.com,France,Paris,Furniture,5,0.20
5,0.752132,55–64,Corporate,3110.18,urgent,Credit Card,3,60,24/04/2026,NaN,353.43,O1034,C022,Carlos Garcia,carlos.garcia@example.com,Netherlands,Amsterdam,Office Supplies,11,0.20
6,0.220499,55–64,Corporate,932.71,urgent,Cash,3,62,24/01/2026,20/01/2026,177.66,O1088,C046,Anna Müller,NaN,Netherlands,Amsterdam,Home,7,0.25
7,0.042570,Under 25,Corporate,203.95,check address,Credit Card,3,19,24/03/2026,22/03/2026,40.79,O1050,C040,Maha Zadeh,maha.zadeh@example.com,France,Paris,Office Supplies,5,0.00
8,0.000000,55–64,Corporate,29.59,ok,Bank Transfer,5,60,NaN,27/01/2026,19.73,O1001,C016,Maha Zadeh,maha.zadeh@example.com,Germany,Mannheim,Office Supplies,2,0.25
9,0.049751,45–54,Corporate,233.36,return risk,Credit Card,1,50,26/03/2026,16/03/2026,58.34,O1092,C012,Ali Reza,ali.reza@example.com,France,Lyon,Furniture,5,0.20


---
## 1. Load & Overview

In [3]:
print(df.dtypes)
df.describe()

Sales_MinMax          float64
Age_Group                 str
Customer_Segment          str
Sales                 float64
Notes                     str
Payment_Method            str
Satisfaction_Score      int64
Age                     int64
Ship_Date                 str
Order_Date                str
Unit_Price            float64
Order_ID                  str
Customer_ID               str
Customer_Name             str
Email                     str
Country                   str
City                      str
Product_Category          str
Quantity                int64
Discount              float64
dtype: object


,Sales_MinMax,Sales,Satisfaction_Score,Age,Unit_Price,Quantity,Discount
count,87.000000,87.000000,91.000000,91.000000,87.000000,91.000000,91.000000
mean,0.303569,1272.951839,3.120879,46.000000,243.324598,5.934066,0.130769
std,0.234472,960.353539,1.162709,14.886982,123.774353,3.147494,0.077735
min,0.000000,29.590000,1.000000,16.000000,13.450000,1.000000,0.000000
25%,0.086512,383.925000,2.000000,36.000000,156.995000,3.000000,0.050000
50%,0.295834,1241.270000,3.000000,45.000000,254.560000,6.000000,0.150000
75%,0.466380,1939.795000,4.000000,56.500000,337.095000,8.000000,0.200000
max,1.000000,4125.400000,5.000000,76.000000,445.070000,12.000000,0.250000


---
## 2. Missing Value Analysis

We check for two types of missingness:
- **True NaN** — pandas reads the cell as null
- **Pseudo-null strings** — values like `"not available"`, `"N/A"`, `""` that are not null but carry no information

In [4]:
true_nulls = df.isnull().sum()
true_null_pct = (df.isnull().mean() * 100).round(2)

null_summary = pd.DataFrame({
    "Missing (NaN)": true_nulls,
    "Missing %": true_null_pct
})

null_summary[null_summary["Missing (NaN)"] > 0].sort_values("Missing %", ascending=False)

,Missing (NaN),Missing %
Email,10,10.99
Order_Date,5,5.49
Sales_MinMax,4,4.40
Unit_Price,4,4.40
Sales,4,4.40
City,3,3.30
Ship_Date,1,1.10


In [5]:
PSEUDO_NULLS = {"not available", "n/a", "na", "none", "null", "-", "unknown", ""}

for col in df.select_dtypes(include="str").columns:
    mask = df[col].str.strip().str.lower().isin(PSEUDO_NULLS)
    if mask.any():
        found = df.loc[mask, col].unique().tolist()
        print(f"{col}: {found}")

Notes: ['none']


---
## 3. Data Type Analysis

In [6]:
df.dtypes

Sales_MinMax          float64
Age_Group                 str
Customer_Segment          str
Sales                 float64
Notes                     str
Payment_Method            str
Satisfaction_Score      int64
Age                     int64
Ship_Date                 str
Order_Date                str
Unit_Price            float64
Order_ID                  str
Customer_ID               str
Customer_Name             str
Email                     str
Country                   str
City                      str
Product_Category          str
Quantity                int64
Discount              float64
dtype: object

In [7]:
def is_clean_number(val):
    if pd.isna(val):
        return True
    try:
        float(str(val).strip())
        return True
    except ValueError:
        return False

df[~df["Unit_Price"].apply(is_clean_number)][["Order_ID", "Customer_Name", "Unit_Price"]]

,Order_ID,Customer_Name,Unit_Price


---
## 4. Duplicate Detection

In [8]:
dupes = df[df.duplicated(subset=["Order_ID"], keep=False)].sort_values("Order_ID")
print(f"Duplicate Order_IDs: {df['Order_ID'].duplicated().sum()}")
print(f"Full row duplicates: {df.duplicated(keep=False).sum()}")
dupes[["Order_ID", "Customer_ID", "Customer_Name", "Order_Date", "Sales"]]

Duplicate Order_IDs: 0
Full row duplicates: 0


,Order_ID,Customer_ID,Customer_Name,Order_Date,Sales


---
## 5. Cardinality Analysis

In [9]:
pd.DataFrame({
    "Unique": df.nunique(),
    "Total": len(df),
    "% Unique": (df.nunique() / len(df) * 100).round(1)
})

,Unique,Total,% Unique
Sales_MinMax,87,91,95.6
Age_Group,6,91,6.6
Customer_Segment,3,91,3.3
Sales,87,91,95.6
Notes,6,91,6.6
Payment_Method,4,91,4.4
Satisfaction_Score,5,91,5.5
Age,41,91,45.1
Ship_Date,63,91,69.2
Order_Date,64,91,70.3


---
## 6. Value Distribution Analysis

In [10]:
EXCLUDE_COLS = ["Order_ID", "Customer_ID", "Customer_Name", "Email",
                "Order_Date", "Ship_Date", "Unit_Price"]

CAT_COLS = [col for col in df.select_dtypes(include="str").columns
            if col not in EXCLUDE_COLS]

for col in CAT_COLS:
    print(df[col].value_counts(dropna=False).rename(col))

Age_Group
55–64       20
65+         18
35–44       18
45–54       15
25–34       11
Under 25     9
Name: Age_Group, dtype: int64
Customer_Segment
Corporate      41
Consumer       26
Home Office    24
Name: Customer_Segment, dtype: int64
Notes
missing phone    22
ok               16
check address    16
return risk      14
urgent           13
none             10
Name: Notes, dtype: int64
Payment_Method
Bank Transfer    36
Credit Card      23
Cash             17
Paypal           15
Name: Payment_Method, dtype: int64
Country
Germany        26
Italy          21
France         17
Spain          15
Netherlands    12
Name: Country, dtype: int64
City
Karlsruhe     14
Paris         10
Mannheim       8
Madrid         8
Milan          8
Rome           8
Lyon           6
Amsterdam      6
Naples         5
Barcelona      5
Heidelberg     4
Rotterdam      4
NaN            3
Valencia       2
Name: City, dtype: int64
Product_Category
Office Supplies    30
Electronics        26
Furniture          18
Hom

In [11]:
# Observed range in describe() is 1 to 5
df[df["Satisfaction_Score"].notna() &
   ~df["Satisfaction_Score"].isin([1, 2, 3, 4, 5])][["Order_ID", "Satisfaction_Score"]]

,Order_ID,Satisfaction_Score


In [12]:
df[df["Quantity"] == 0][["Order_ID", "Customer_Name", "Product_Category", "Quantity", "Unit_Price", "Sales"]]

,Order_ID,Customer_Name,Product_Category,Quantity,Unit_Price,Sales


---
## 7. Text Format & Inconsistency Analysis

In [13]:
# Leading/trailing whitespace in Customer_Name
df[df["Customer_Name"].str.strip() != df["Customer_Name"]][["Order_ID", "Customer_Name"]]

,Order_ID,Customer_Name


In [14]:
# Casing variants grouped by normalised name
name_groups = (df.groupby(df["Customer_Name"].str.strip().str.lower())["Customer_Name"]
                 .unique())
name_groups.index.name = "Normalised"
name_groups[name_groups.apply(len) > 1].reset_index()

,Normalised,Customer_Name


In [15]:
EMAIL_PATTERN = r'^[\w.+-]+@[\w-]+\.[\w.]+$'

df[df["Email"].notna() &
   ~df["Email"].str.match(EMAIL_PATTERN)][["Order_ID", "Customer_Name", "Email"]]

,Order_ID,Customer_Name,Email


---
## 8. Date Format Analysis

In [16]:
def detect_date_format(val):
    if pd.isna(val) or str(val).strip().lower() in ["not available", ""]:
        return "null/pseudo-null"
    val = str(val).strip()
    patterns = [
        (r'^\d{4}/\d{2}/\d{2}$',   'YYYY/MM/DD'),
        (r'^\d{4}-\d{2}-\d{2}$',   'YYYY-MM-DD'),
        (r'^\d{2}/\d{2}/\d{4}$',   'DD/MM/YYYY'),
        (r'^\d{2}-\d{2}-\d{4}$',   'MM-DD-YYYY'),
        (r'^\d{2}\.\d{2}\.\d{4}$', 'DD.MM.YYYY'),
    ]
    for pat, label in patterns:
        if re.match(pat, val):
            return label
    return f'unknown: {val}'

pd.DataFrame({col: df[col].apply(detect_date_format).value_counts(dropna=False)
              for col in ["Order_Date", "Ship_Date"]})

,Order_Date,Ship_Date
DD/MM/YYYY,86,90
null/pseudo-null,5,1


In [17]:
def try_parse_date(val):
    if pd.isna(val) or str(val).strip().lower() in ["not available", ""]:
        return pd.NaT
    for fmt in ["%Y/%m/%d", "%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%d.%m.%Y"]:
        try:
            return pd.to_datetime(str(val).strip(), format=fmt)
        except ValueError:
            continue
    return pd.NaT

order_dates = df["Order_Date"].apply(try_parse_date)
ship_dates  = df["Ship_Date"].apply(try_parse_date)

bad_seq = ship_dates.notna() & order_dates.notna() & (ship_dates < order_dates)
df.loc[bad_seq, ["Order_ID", "Customer_Name", "Order_Date", "Ship_Date"]]

,Order_ID,Customer_Name,Order_Date,Ship_Date


In [18]:
same_day = ship_dates.notna() & order_dates.notna() & (ship_dates == order_dates)
df.loc[same_day, ["Order_ID", "Customer_Name", "Order_Date", "Ship_Date"]]

,Order_ID,Customer_Name,Order_Date,Ship_Date
36,O1014,Julia Fischer,08/04/2026,08/04/2026
45,O1041,Laura Rossi,06/01/2026,06/01/2026


---
## 9. Numeric Validity & Outlier Analysis

In [19]:
df[(df["Age"] > 120) | (df["Age"] < 0)][["Order_ID", "Customer_Name", "Age"]]

,Order_ID,Customer_Name,Age


In [20]:
try:
    df[df["Unit_Price"].str.contains("9999", na=False)][["Order_ID", "Customer_Name", "Unit_Price", "Quantity"]]
except Exception as e:
    print("No 9999")

No 9999


In [21]:
Q1, Q3 = df["Sales"].quantile(0.25), df["Sales"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

print(f"Lower fence: {lower:.2f} | Upper fence: {upper:.2f}")
df[(df["Sales"] < lower) | (df["Sales"] > upper)][["Order_ID", "Customer_Name", "Unit_Price", "Quantity", "Discount", "Sales"]]

Lower fence: -1949.88 | Upper fence: 4273.60


,Order_ID,Customer_Name,Unit_Price,Quantity,Discount,Sales


---
## 10. Structure Analysis

In [22]:
norm_counts = df.groupby("Customer_ID")["Customer_Name"].apply(
    lambda x: x.str.strip().str.lower().nunique()
)
violations = norm_counts[norm_counts > 1].index
df[df["Customer_ID"].isin(violations)].groupby("Customer_ID")["Customer_Name"].unique()

Customer_ID
C003                          [Laura Rossi, Luca Bianchi]
C004                            [Marie Dubois, Tom Brown]
C005                [Maha Zadeh, Nina Weber, Anna Müller]
C006                         [Luca Bianchi, Marie Dubois]
C009                           [Luca Bianchi, Nina Weber]
C010                           [Marie Dubois, Nina Weber]
C012                              [Ali Reza, Anna Müller]
C014            [Nina Weber, Carlos Garcia, Marie Dubois]
C017                         [Carlos Garcia, Emma Wilson]
C018                          [Carlos Garcia, Maha Zadeh]
C019    [Ali Reza, John Doe, Marie Dubois, Julia Fischer]
C020                           [Mehdi Karimi, Maha Zadeh]
C021                            [Mehdi Karimi, Tom Brown]
C022               [Carlos Garcia, John Doe, Anna Müller]
C024                             [Mehdi Karimi, John Doe]
C029               [John Doe, Julia Fischer, Anna Müller]
C030                            [Laura Rossi, Nina Weber]
C0

In [23]:
def clean_price(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace('€','').replace('$','').replace('£','').replace(',','.')
    try:
        return float(s)
    except:
        return np.nan

price_clean = df["Unit_Price"].apply(clean_price)
expected_sales = (price_clean * df["Quantity"] * (1 - df["Discount"])).round(2)
diff = (df["Sales"] - expected_sales).abs()

inconsistent = df[diff > 0.05].copy()
inconsistent["expected_sales"] = expected_sales[diff > 0.05]
inconsistent[["Order_ID", "Unit_Price", "Quantity", "Discount", "expected_sales", "Sales"]]

,Order_ID,Unit_Price,Quantity,Discount,expected_sales,Sales
